# Semana 2: Introducción a Spark con Scala

## Contexto del Ejercicio: Mi Porfolio como Data Engineer

Este notebook forma parte de la actividad **"Mi porfolio como Data Engineer"** descrita en el syllabus (páginas 18-19). 

El objetivo es continuar la construcción de tu **base de conocimiento** (porfolio). Puedes utilizar este notebook como:
*   **Guía**: Para entender la arquitectura de Spark y su API.
*   **Base**: Para experimentar con diferentes transformaciones y acciones.
*   **Complemento**: A tu repositorio de código en GitHub.

---

En esta segunda semana, nos adentraremos en el desarrollo de aplicaciones distribuidas utilizando Apache Spark. Veremos los conceptos fundamentales, trabajaramos con RDDs (Spark Core) y DataFrames (Spark SQL).

In [1]:
import $ivy.`com.nrinaudo::kantan.csv:0.6.1`
import $ivy.`com.nrinaudo::kantan.csv-generic:0.6.1`
import org.apache.spark.sql.SparkSession

case class Pasajero(
  PassengerId: Int,
  Survived: Int,
  Pclass: Int,
  Name: String,
  Sex: String,
  Age: Option[Double],
  SibSp: Int,
  Parch: Int,
  Ticket: String,
  Fare: Double,
  Cabin: Option[String],
  Embarked: Option[String]
)


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
defined class Pasajero

In [2]:

// Inicialización de SparkSession (El punto de entrada a Spark)
val spark = SparkSession.builder()
  .appName("Semana2_Porfolio")
  .master("local[*]") // Ejecutar localmente usando todos los cores disponibles
  //.master("spark://spark-master:7077") // Si activas este modo obtendrás algunos errores por la integración de Ammonite y Spark
  // Memoria del Driver (donde se recolectan los resultados de .collect())
  .config("spark.driver.memory", "2g") 
  // Memoria de cada Executor
  .config("spark.executor.memory", "2g")
  // Memoria adicional por encima del heap (útil para evitar errores de overhead)
  .config("spark.executor.memoryOverhead", "512m")
  //.config("deploy-mode","client")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR") // Reducir el ruido en los logs

println(s"Spark Version: ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/18 20:07:07 INFO SparkContext: Running Spark version 4.1.1
26/02/18 20:07:07 INFO SparkContext: OS info Linux, 6.6.87.2-microsoft-standard-WSL2, amd64
26/02/18 20:07:07 INFO SparkContext: Java version 17.0.18+8-Ubuntu-122.04.1
26/02/18 20:07:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/18 20:07:07 INFO ResourceUtils: ==============================================================
26/02/18 20:07:07 INFO ResourceUtils: No custom resources configured for spark.driver.
26/02/18 20:07:07 INFO ResourceUtils: ==============================================================
26/02/18 20:07:07 INFO SparkContext: Submitted application: Semana2_Porfolio
26/02/18 20:07:07 INFO SecurityManager: Changing view acls to: jovyan
26/02/18 20:07:07 INFO SecurityManager: Changing modify acls to: jovyan
26/02/18 20:07:07 INFO SecurityMana

Spark Version: 4.1.1


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@20310188

In [3]:
// Obtenemos el estado de los ejecutores
// Retorna un Map: "host:puerto" -> (Memoria Total, Memoria Libre)
val status = spark.sparkContext.getExecutorMemoryStatus

val hosts = status.keys.toSeq
val totalMem = status.values.map(_._1 / (1024 * 1024)).toSeq // Convertir a MB

println(s"Nodos activos detectados: ${hosts.size}")

Nodos activos detectados: 1


status: collection.Map[String, (Long, Long)] = Map(
  "2a57f1c00c94:35157" -> (890870169L, 890870169L)
)
hosts: Seq[String] = List("2a57f1c00c94:35157")
totalMem: Seq[Long] = List(849L)

In [3]:
// En el Driver (Jupyter)
println(System.getProperty("java.version"))

// En los Workers
spark.sparkContext.parallelize(Seq(1)).map(_ => System.getProperty("java.version")).collect().foreach(println)

17.0.18
17.0.18


## 2. Introducción al módulo Spark Core (RDDs)

RDD (Resilient Distributed Dataset) es la abstracción fundamental de Spark. Representa una colección inmutable de objetos distribuida y tolerante a fallos.

### Transformaciones vs Acciones
*   **Transformaciones (Lazy)**: Crean un nuevo RDD a partir de uno existente (ej. `map`, `filter`). No se ejecutan inmediatamente.
*   **Acciones**: Disparan la computación y devuelven un resultado al Driver o guardan datos (ej. `count`, `collect`, `saveAsTextFile`).

In [4]:
// Ejemplo Spark Core: Procesamiento de texto básico con RDDs
val datos = Seq("Spark es rapido", "Spark es genial", "Scala y Spark", "Big Data es el futuro")

// 1. Crear RDD paralelizando una colección existente
val rdd = spark.sparkContext.parallelize(datos)

// 2. Transformaciones
val palabrasRDD = rdd
  .flatMap(linea => linea.split(" ")) // Dividir frases en palabras
  .map(palabra => palabra.toLowerCase) // Convertir a minúsculas
  .filter(palabra => palabra.contains("s")) // Filtrar palabras que contienen 's'

// 3. Acción (Solo aquí se ejecuta el procesamiento)
val resultado = palabrasRDD.collect()

println("Palabras con 's':")
resultado.foreach(println)


Palabras con 's':
spark
es
spark
es
scala
spark
es


datos: Seq[String] = List(
  "Spark es rapido",
  "Spark es genial",
  "Scala y Spark",
  "Big Data es el futuro"
)
rdd: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[2] at parallelize at cmd4.sc:5
palabrasRDD: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[5] at filter at cmd4.sc:11
resultado: Array[String] = Array(
  "spark",
  "es",
  "spark",
  "es",
  "scala",
  "spark",
  "es"
)

## 3. Introducción al módulo Spark SQL (DataFrames)

Spark SQL permite consultar datos estructurados. El DataFrame es la abstracción principal aquí: es como una tabla en una base de datos relacional o un DataFrame en pandas, pero distribuido.

Los DataFrames utilizan el **Catalyst Optimizer** para optimizar automáticamente las consultas.

In [5]:
import spark.implicits._ // Import necesario para conversiones implícitas a DF

// Ejemplo Spark SQL: DataFrames

// 1. Crear DataFrame desde una secuencia de tuplas
val personas = Seq(
  ("Alice", 28, "Data Engineer"),
  ("Bob", 35, "Data Scientist"),
  ("Charlie", 23, "Data Analyst"),
  ("David", 42, "Data Engineer")
)

val df = personas.toDF("nombre", "edad", "rol")

// 2. Mostrar el esquema y los datos
df.printSchema()
df.show()

// 3. Consultas usando API de DataFrame
println("Data Engineers mayores de 25:")
df.filter($"rol" === "Data Engineer" && $"edad" > 25)
  .select("nombre", "edad")
  .show()

// 4. Agregaciones
println("Edad promedio por rol:")
df.groupBy("rol")
  .avg("edad")
  .show()

root
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = false)
 |-- rol: string (nullable = true)

+-------+----+--------------+
| nombre|edad|           rol|
+-------+----+--------------+
|  Alice|  28| Data Engineer|
|    Bob|  35|Data Scientist|
|Charlie|  23|  Data Analyst|
|  David|  42| Data Engineer|
+-------+----+--------------+

Data Engineers mayores de 25:
+------+----+
|nombre|edad|
+------+----+
| Alice|  28|
| David|  42|
+------+----+

Edad promedio por rol:
+--------------+---------+
|           rol|avg(edad)|
+--------------+---------+
| Data Engineer|     35.0|
|Data Scientist|     35.0|
|  Data Analyst|     23.0|
+--------------+---------+



import spark.implicits._
personas: Seq[(String, Int, String)] = List(
  ("Alice", 28, "Data Engineer"),
  ("Bob", 35, "Data Scientist"),
  ("Charlie", 23, "Data Analyst"),
  ("David", 42, "Data Engineer")
)
df: org.apache.spark.sql.package.DataFrame = [nombre: string, edad: int ... 1 more field]

In [6]:
// 5. Consultas SQL estándar
// Registramos el DataFrame como una vista temporal
df.createOrReplaceTempView("personas_view")

val sqlDF = spark.sql("SELECT rol, count(*) as total FROM personas_view GROUP BY rol")
sqlDF.show()

+--------------+-----+
|           rol|total|
+--------------+-----+
| Data Engineer|    2|
|Data Scientist|    1|
|  Data Analyst|    1|
+--------------+-----+



sqlDF: org.apache.spark.sql.package.DataFrame = [rol: string, total: bigint]

## 4. Ejercicios Prácticos (Sin resolver)

Usa estos ejercicios como base para practicar y documentar en tu porfolio.

### Ejercicio 1: Manipulación de DataFrames
Crea un DataFrame a partir de una lista de productos (nombre, precio, stock). Luego:
1. Añade una columna `valor_inventario` (precio * stock).
2. Filtra los productos que tengan un stock menor a 10.
3. Muestra el resultado.

In [10]:
import org.apache.spark.sql.functions._

import spark.implicits._ // Import necesario para conversiones implícitas a DF

// Definir datos y crear DataFrame
val productos = Seq(("Laptop", 1200.0, 5), 
                    ("Mouse", 25.0, 20),
                    ("Keyboard", 29.0, 20),
                    ("Screen", 325.0, 11),
                    ("DockStation", 225.0, 2))
// Creamos DataFrame
val dfProductos = productos.toDF("nombre", "precio", "stock")
// Transformaciones
// 2. Añade una columna valor_inventario (precio * stock).
val df_valorInventario = dfProductos.withColumn("valor_inventario", col("nombre") * col("precio"))
// Filtrar los productos con stock menor a 10 y lo guardo en otro df
val dfResultado = dfProductos.filter(col("stock") < 10)

dfResultado.show()


// dfResultado.show()

+-----------+------+-----+
|     nombre|precio|stock|
+-----------+------+-----+
|     Laptop|1200.0|    5|
|DockStation| 225.0|    2|
+-----------+------+-----+



import org.apache.spark.sql.functions._
import spark.implicits._
productos: Seq[(String, Double, Int)] = List(
  ("Laptop", 1200.0, 5),
  ("Mouse", 25.0, 20),
  ("Keyboard", 29.0, 20),
  ("Screen", 325.0, 11),
  ("DockStation", 225.0, 2)
)
dfProductos: org.apache.spark.sql.package.DataFrame = [nombre: string, precio: double ... 1 more field]
df_valorInventario: org.apache.spark.sql.package.DataFrame = [nombre: string, precio: double ... 2 more fields]
dfResultado: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [nombre: string, precio: double ... 1 more field]

### Ejercicio 2: Consultas SQL
Registra el DataFrame de productos anterior como una vista temporal y realiza una consulta SQL que devuelva el precio medio de los productos.

In [11]:
import org.apache.spark.sql.functions._

import spark.implicits._ // Import necesario para conversiones implícitas a DF

// Registrar vista y ejecutar SQL
dfProductos.createOrReplaceTempView("productos_temp")

// Realizamos la select para obtener el precio medio
val precioMedio = spark.sql("SELECT AVG(precio) AS precio_medio FROM productos_temp")

precioMedio.show()

+------------+
|precio_medio|
+------------+
|       360.8|
+------------+



import org.apache.spark.sql.functions._
import spark.implicits._
precioMedio: org.apache.spark.sql.package.DataFrame = [precio_medio: double]

# Ejercicio Propio#
## transformacion carga de un fichero CSV, con formateo automatico ##

El ejercicio consiste en:
1. cargar los datos del pasaje del Titanic de un repositorio publico.
2. Tranformar los datos 
4. Cargar un dataFrame
5. Cargar un dataSet
6. Cargarlos en una tabla temporal 
7. Obtener la edad media del pasaje

In [8]:
// Necesito las versiones para cargar las libreria s correctas
scala.util.Properties.versionString
spark.version


res8_0: String = "version 2.13.17"
res8_1: String = "4.1.1"

In [17]:
/** Lo muevo al principio
import $ivy.`com.nrinaudo::kantan.csv:0.6.1`
import $ivy.`com.nrinaudo::kantan.csv-generic:0.6.1`
*/

import kantan.csv._
import kantan.csv.ops._
import kantan.csv.generic._
import scala.io.Source


/** Lo muevo al principio 
case class Pasajero(
  PassengerId: Int,
  Survived: Int,
  Pclass: Int,
  Name: String,
  Sex: String,
  Age: Option[Double],
  SibSp: Int,
  Parch: Int,
  Ticket: String,
  Fare: Double,
  Cabin: Option[String],
  Embarked: Option[String]
)
*/

// 1. cargar los datos del pasaje del Titanic de un repositorio publico.
val url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
val contenido = Source.fromURL(url).mkString


val datos = contenido.asCsvReader[Pasajero](rfc.withHeader).toList
// 2. Tranformar los datos 
val pasajeros = datos.collect { case Right(p) => p }

println(datos.take(5))

// 4. Cargar un dataFrame
// Pasamos los datos a un dataframe no tipado.
val df_pasaje = spark.createDataFrame(pasajeros)
df_pasaje.show(5)
df_pasaje.printSchema()

// 5. Cargar un dataSet
// Pasamos los datos a un data set
import spark.implicits._
val ds_pasaje = spark.createDataset(pasajeros)
ds_pasaje.show(5)
ds_pasaje.printSchema()

// Ahora vamos a cargar los datos en un Spark SQL
import spark.implicits._ // Import necesario para conversiones implícitas a DF

// 6. Cargarlos en una tabla temporal 
// Registrar vista y ejecutar SQL
df_pasaje.createOrReplaceTempView("pasaje_temp")

// 7. Obtener la edad media del pasaje
// Realizamos la select para obtener la edad media del pasaje
val edadMediaDF = spark.sql("SELECT AVG(age) AS EdadMedia FROM pasaje_temp")

// Extraemos el valor como Double y
val edadMedia = edadMediaDF.first().getAs[Double]("EdadMedia")

// Impimimos el dato sin decimales
println(f"La edad media del pasaje es: $edadMedia%.0f")




List(Right(Pasajero(1,0,3,Braund, Mr. Owen Harris,male,Some(22.0),1,0,A/5 21171,7.25,None,Some(S))), Right(Pasajero(2,1,1,Cumings, Mrs. John Bradley (Florence Briggs Thayer),female,Some(38.0),1,0,PC 17599,71.2833,Some(C85),Some(C))), Right(Pasajero(3,1,3,Heikkinen, Miss. Laina,female,Some(26.0),0,0,STON/O2. 3101282,7.925,None,Some(S))), Right(Pasajero(4,1,1,Futrelle, Mrs. Jacques Heath (Lily May Peel),female,Some(35.0),1,0,113803,53.1,Some(C123),Some(S))), Right(Pasajero(5,0,3,Allen, Mr. William Henry,male,Some(35.0),0,0,373450,8.05,None,Some(S))))
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| 

import kantan.csv._
import kantan.csv.ops._
import kantan.csv.generic._
import scala.io.Source
url: String = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
contenido: String = """PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.05,,S
6,0,3,"Moran, Mr. James",male,,0,0,330877,8.4583,,Q
7,0,1,"McCarthy, Mr. Timothy J",male,54,0,0,17463,51.8625,E46,S
8,0,3,"Palsson, Master. Gosta Leonard",male,2,3,1,349909,21.075,,S
9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27,0,2,347742,11.1333,,S
10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14,1,0,

In [18]:
spark.stop()